# Beyond Distance — Ball-Carrying Kinematics Pipeline

This notebook is a direct conversion of the `final_code/` project into a single, ordered,
runnable notebook. It reflects the **corrected** pipeline behind the "Beyond Distance"
research (La Liga 2020/21 + Ligue 1 2021/22, StatsBomb open data, 16,472 real carries) —
including the forward-progress bug fix documented in the thesis (Section 3.6), not the
earlier draft versions.

Everything runs on **StatsBomb's free open-data release**
(https://github.com/statsbomb/open-data). No API key or paid data access is needed.

## Pipeline order

Run the sections below in sequence — each step's output feeds the next.

| # | Section | What it does | Produces |
|---|--------|--------------|----------|
| 1 | Download raw data | Downloads real match event + 360° freeze-frame JSON files from StatsBomb open data, in rate-limit-safe batches. | `events/*.json`, `threesixty/*.json` per season |
| 2 | Build base carries | Parses raw match files once into a cached table of carry-level features (reused by the grid searches below). | `base_carries_s1.pkl` |
| 3 | 1-D gap grid search | **Initial specification.** Validates the depth-only line-clustering gap threshold via held-out train/val/test grid search — thesis Table 3.4. | `gap_grid_search.json` |
| 4 | Spatial library | **Core geometry module.** DBSCAN clustering, convex-hull construction, forward-progress-only path intersection, static Voronoi pitch control. This is the corrected version — see "Known issue, fixed" below. | — (library) |
| 5 | 2-D grid search | **2-D upgrade validation.** Grid-searches lateral-distance weight and clustering gap for the genuine 2-D method — thesis Table 3.5. | `data/spatial_params_final.json` |
| 6 | Main pipeline | Re-parses all real match files using the validated 2-D parameters: progressive-carry flags, 2-D line-breaking, static pitch control, multi-phase (possession-chain) downstream xG. Single source of truth for all reported numbers. | `data/carries_final.csv` |
| 7 | Formal statistics | OLS (robust SE) for pitch-control-gained, logistic regression for line-breaking (6-feature baseline), player fixed-effects logistic regression with full p-values/CIs, bootstrap 95% CIs for classifier AUC — thesis Tables 4.3–4.7. | `data/inference_results.json` |
| 8 | Dashboard v2 | Season 1 descriptive dashboard (speed distribution, line-break rates, feature importance) — thesis Figure 4.1. | `real_kinematic_dashboard_v2.png` |
| 9 | Dashboard v3 | Grid-search curve, xG-linkage comparison, Season 2 replication, AUC comparison — addendum figure. | `dashboard_v3.png` |
| 10 | Spatial example | Renders one real carry's convex hulls and Voronoi pitch-control grid — thesis Figure 3.2/3.3. | `spatial_example.png` |
| 11 | Forest plot | Player fixed-effects coefficients with 95% CIs, both seasons — thesis Figure 4.2. | `forest_plot_fixed_effects.png` |

## `data/` — final outputs already generated

This notebook expects a `data/` folder next to it containing the files shipped in the
original project:

- `carries_final.csv` — the master dataset: 16,472 real carries, both seasons, every metric.
- `inference_results.json` — every regression table's coefficients, SEs, p-values, CIs.
- `spatial_params_final.json` — the validated 2-D clustering parameters (y_weight=0.35, eps=6).
- `gap_grid_search.json` — the 1-D grid search results (superseded by the 2-D method, kept for the methodology narrative).

**If you just want to re-run the analysis without re-downloading ~400MB of match data,
skip straight to Step 7** — steps 1–6 are only needed to regenerate `carries_final.csv`
from scratch or to re-validate the spatial parameters yourself.

## Known issue, fixed (keep this in your presentation notes)

An early version of `spatial_lib.py`'s `carry_path_crosses_hulls()` had no directionality
check, so backward/lateral carries could register as line-breaking by geometrically
grazing a hull (affected 31% of carries, 22.8% of which were wrongly flagged). The current
version requires `end[0] > start[0]` before any hull-intersection test runs. Every grid
search, dataset, and statistical result in this notebook was regenerated after that fix.

## Data source

StatsBomb Open Data, competition_id 11 / season_id 90 (La Liga 2020/21) and
competition_id 7 / season_id 108 (Ligue 1 2021/22). https://github.com/statsbomb/open-data


## Setup

Install dependencies and set shared paths. Run this cell first.

In [ ]:
# Uncomment to install dependencies
# %pip install pandas numpy scipy scikit-learn shapely statsmodels matplotlib --quiet

import os

# Where the pre-generated data files (carries_final.csv, inference_results.json, ...) live
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

print("Data directory:", os.path.abspath(DATA_DIR))


## Step 1 — Download raw match data (`redownload.py`)

Downloads real match event + 360° freeze-frame JSON files from StatsBomb open data, in
rate-limit-safe batches.

**Optional.** Only needed if you want to regenerate everything from scratch. This can take a
while and pulls down several hundred MB of JSON. If you already have `events/` and
`threesixty/` folders (or you're starting from `data/carries_final.csv` at Step 7), skip
this section.

Original usage: `python3 redownload.py <matches_file> <events_dir> <threesixty_dir> <start_idx> <batch_size>`


In [ ]:
import json, time, urllib.request, urllib.error, os, sys

BASE = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"

def fetch(url, path, max_retries=8):
    if os.path.exists(path) and os.path.getsize(path) > 100:
        return True
    delay = 3
    for attempt in range(max_retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "research-script/1.0"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = resp.read()
            with open(path, "wb") as f:
                f.write(data)
            json.loads(data)
            return True
        except (urllib.error.HTTPError, urllib.error.URLError, json.JSONDecodeError) as e:
            print(f"  retry {attempt+1} for {path}: {e}")
            time.sleep(delay); delay = min(delay*1.7, 30)
    return False

def download_batch(matches_file, events_dir, threesixty_dir, start, limit):
    matches = json.load(open(matches_file))
    match_ids = [m["match_id"] for m in matches]
    batch = match_ids[start:start+limit]
    print(f"{len(batch)} matches to fetch this run")
    for i, mid in enumerate(batch):
        fetch(f"{BASE}/events/{mid}.json", f"{events_dir}/{mid}.json"); time.sleep(0.35)
        fetch(f"{BASE}/three-sixty/{mid}.json", f"{threesixty_dir}/{mid}.json"); time.sleep(0.35)
        print(f"[{i+1}/{len(batch)}] match {mid} done", flush=True)

# Edit these and run to fetch a batch of matches, e.g. for La Liga 2020/21:
#   matches_file  = path to a StatsBomb "matches" JSON (competition_id 11, season_id 90)
#   events_dir    = "events"
#   threesixty_dir = "threesixty"
# matches_file, events_dir, threesixty_dir, start, limit = "matches_s1.json", "events", "threesixty", 0, 20
# download_batch(matches_file, events_dir, threesixty_dir, start, limit)


## Step 2 — Build base carries (`build_base.py`)

Parses raw match files once into a cached table of carry-level features (used repeatedly
by the grid searches below, so this avoids re-parsing JSON for every parameter tried).

Requires `events/` and `threesixty/` from Step 1.


In [ ]:
import json, glob, math
import pandas as pd

MID_ATT_POSITIONS = {
    "Left Center Midfield", "Right Center Midfield", "Center Defensive Midfield",
    "Left Defensive Midfield", "Right Defensive Midfield", "Left Midfield", "Right Midfield",
    "Center Attacking Midfield", "Left Attacking Midfield", "Right Attacking Midfield",
    "Left Wing", "Right Wing", "Center Forward", "Left Center Forward", "Right Center Forward",
}
YARDS_TO_M = 0.9144

def dist_yd(a, b):
    return math.hypot(a[0]-b[0], a[1]-b[1])

def build_base(events_dir, threesixty_dir):
    rows = []
    for ef in sorted(glob.glob(f"{events_dir}/*.json")):
        match_id = ef.split("/")[-1].replace(".json", "")
        events = json.load(open(ef))
        try:
            frames = json.load(open(f"{threesixty_dir}/{match_id}.json"))
            frame_by_event = {f["event_uuid"]: f for f in frames}
        except FileNotFoundError:
            frame_by_event = {}
        for e in events:
            if e["type"]["name"] != "Carry": continue
            pos = e.get("position", {}).get("name")
            if pos not in MID_ATT_POSITIONS: continue
            if "location" not in e or "carry" not in e or "end_location" not in e["carry"]: continue
            start, end = e["location"], e["carry"]["end_location"]
            duration = e.get("duration")
            if duration is None or duration <= 0: continue
            dist_yards = dist_yd(start, end)
            dist_m = dist_yards * YARDS_TO_M
            speed_ms = dist_m / duration
            if dist_yards < 3.0 or duration < 0.4 or speed_ms > 12.0: continue
            frame = frame_by_event.get(e["id"])
            if not (frame and frame.get("freeze_frame")): continue
            opp_pts = [pl["location"] for pl in frame["freeze_frame"] if not pl.get("teammate", True) and not pl.get("keeper", False)]
            if not opp_pts: continue
            defender_prox_m = min(dist_yd(start, o) * YARDS_TO_M for o in opp_pts)
            rows.append({
                "match_id": match_id, "start": start, "end": end,
                "carry_distance_m": dist_m, "carry_duration_s": duration, "carry_speed_ms": speed_ms,
                "defender_proximity_m": defender_prox_m,
                "under_pressure": int(bool(e.get("under_pressure", False))),
                "opp_pts": opp_pts,
            })
    return pd.DataFrame(rows)

# df1 = build_base("events", "threesixty")
# print("Season 1 base carries:", len(df1))
# df1.to_pickle("base_carries_s1.pkl")


## Step 3 — 1-D gap grid search (`grid_search_gap.py`)

**Initial (1-D) specification.** Validates the depth-only line-clustering gap threshold via
held-out train/val/test grid search. This is the *first* version of the line-break metric
(thesis Table 3.4) — superseded by the 2-D method in Step 5, but kept here for the
methodology narrative.

Requires `events/` and `threesixty/` from Step 1. Writes `gap_grid_search.json` to the
current directory (already provided at `data/gap_grid_search.json`).


In [ ]:
# NOTE: this cell expects events/threesixty from Step 1 and will error without them.
import os
import json, glob, math
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

MID_ATT_POSITIONS = {
    "Left Center Midfield", "Right Center Midfield", "Center Defensive Midfield",
    "Left Defensive Midfield", "Right Defensive Midfield", "Left Midfield", "Right Midfield",
    "Center Attacking Midfield", "Left Attacking Midfield", "Right Attacking Midfield",
    "Left Wing", "Right Wing", "Center Forward", "Left Center Forward", "Right Center Forward",
}
YARDS_TO_M = 0.9144

def dist_yd(a, b):
    return math.hypot(a[0]-b[0], a[1]-b[1])

def cluster_lines(xs, gap):
    if not xs: return []
    xs = sorted(xs)
    lines, current = [], [xs[0]]
    for x in xs[1:]:
        if x - current[-1] > gap:
            lines.append(current); current = [x]
        else:
            current.append(x)
    lines.append(current)
    return [sum(c)/len(c) for c in lines]

# ---- Build the base carry table once (features + raw opponent x-lists), gap applied later ----
base_rows = []
for ef in sorted(glob.glob("events/*.json")):
    match_id = ef.split("/")[-1].replace(".json", "")
    events = json.load(open(ef))
    try:
        frames = json.load(open(f"threesixty/{match_id}.json"))
        frame_by_event = {f["event_uuid"]: f for f in frames}
    except FileNotFoundError:
        frame_by_event = {}
    for e in events:
        if e["type"]["name"] != "Carry": continue
        pos = e.get("position", {}).get("name")
        if pos not in MID_ATT_POSITIONS: continue
        if "location" not in e or "carry" not in e or "end_location" not in e["carry"]: continue
        start, end = e["location"], e["carry"]["end_location"]
        duration = e.get("duration")
        if duration is None or duration <= 0: continue
        dist_yards = dist_yd(start, end)
        dist_m = dist_yards * YARDS_TO_M
        speed_ms = dist_m / duration
        if dist_yards < 3.0 or duration < 0.4 or speed_ms > 12.0: continue
        frame = frame_by_event.get(e["id"])
        if not (frame and frame.get("freeze_frame")): continue
        opp_players = [pl for pl in frame["freeze_frame"] if not pl.get("teammate", True)]
        opp_locs = [pl["location"] for pl in opp_players]
        outfield_x = [pl["location"][0] for pl in opp_players if not pl.get("keeper", False)]
        if not opp_locs: continue
        defender_prox_m = min(dist_yd(start, o) * YARDS_TO_M for o in opp_locs)
        base_rows.append({
            "carry_distance_m": dist_m, "carry_duration_s": duration, "carry_speed_ms": speed_ms,
            "defender_proximity_m": defender_prox_m, "under_pressure": int(bool(e.get("under_pressure", False))),
            "start_x": start[0], "end_x": end[0], "outfield_x": outfield_x,
        })

df = pd.DataFrame(base_rows)
print(f"Base carries with 360 features: {len(df)}")

feat_cols = ["carry_distance_m", "carry_speed_ms", "defender_proximity_m", "under_pressure"]

# 60/20/20 train/val/test split, fixed once so every gap is evaluated on the SAME split
idx = np.arange(len(df))
idx_train, idx_temp = train_test_split(idx, test_size=0.4, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42)

gap_grid = [3, 4, 5, 6, 7, 8, 9, 10]
val_results = []
for gap in gap_grid:
    lines_broken = []
    for _, row in df.iterrows():
        if row["end_x"] > row["start_x"]:
            lo, hi = row["start_x"], row["end_x"]
            line_positions = cluster_lines(row["outfield_x"], gap)
            broken = sum(1 for lx in line_positions if lo < lx <= hi)
        else:
            broken = 0
        lines_broken.append(int(broken >= 1))
    y = np.array(lines_broken)
    rate = y.mean()

    X = df[feat_cols].values
    Xtr, ytr = X[idx_train], y[idx_train]
    Xval, yval = X[idx_val], y[idx_val]
    gb = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42)
    gb.fit(Xtr, ytr)
    val_auc = roc_auc_score(yval, gb.predict_proba(Xval)[:, 1])
    val_results.append({"gap_yd": gap, "gap_m": round(gap*YARDS_TO_M,2), "line_break_rate": round(100*rate,2), "val_auc": round(val_auc,4)})
    print(f"gap={gap}yd ({gap*YARDS_TO_M:.1f}m)  rate={100*rate:.2f}%  val_AUC={val_auc:.4f}")

best = max(val_results, key=lambda r: r["val_auc"])
print(f"\nBest gap by validation AUC: {best}")

# Final test-set evaluation at the chosen gap (never touched during selection)
chosen_gap = best["gap_yd"]
lines_broken = []
for _, row in df.iterrows():
    if row["end_x"] > row["start_x"]:
        lo, hi = row["start_x"], row["end_x"]
        line_positions = cluster_lines(row["outfield_x"], chosen_gap)
        broken = sum(1 for lx in line_positions if lo < lx <= hi)
    else:
        broken = 0
    lines_broken.append(int(broken >= 1))
y = np.array(lines_broken)
X = df[feat_cols].values
Xtrval = np.concatenate([X[idx_train], X[idx_val]])
ytrval = np.concatenate([y[idx_train], y[idx_val]])
gb_final = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42)
gb_final.fit(Xtrval, ytrval)
test_auc = roc_auc_score(y[idx_test], gb_final.predict_proba(X[idx_test])[:, 1])
print(f"Final held-out TEST AUC at gap={chosen_gap}yd: {test_auc:.4f}")

json.dump({"grid": val_results, "chosen_gap_yd": chosen_gap, "chosen_gap_m": round(chosen_gap*YARDS_TO_M,2), "test_auc": round(test_auc,4)},
          open(os.path.join(DATA_DIR, "gap_grid_search.json"), "w"), indent=2)


## Step 4 — Core spatial library (`spatial_lib.py`)

DBSCAN clustering, convex-hull construction, forward-progress-only path intersection, and
static Voronoi pitch control. This is the corrected version (see "Known issue, fixed"
above) — used by Steps 5, 6, 9 and 10.

We write it to a real `spatial_lib.py` file on disk and import it as a module (rather than
inlining the functions), because later steps mutate `spatial_lib.Y_WEIGHT` as a module
attribute — exactly as the original scripts do.


In [ ]:
spatial_lib_source = r"""
import numpy as np
from sklearn.cluster import DBSCAN
from scipy.spatial import cKDTree
from shapely.geometry import LineString, Point, MultiPoint
from shapely.ops import unary_union

Y_WEIGHT = 0.25  # anisotropic scaling: 1 yard of lateral (y) separation counts as much
                 # less "different line" evidence than 1 yard of depth (x) separation

def cluster_lines_2d(points_xy, eps_x_equivalent):
    """DBSCAN on (x, y*Y_WEIGHT) so clustering is primarily depth-driven (tactically
    correct: a 'line' spans the pitch width but is grouped by how far downfield it is),
    while still using real 2-D positions (unlike the old x-only 1-D method)."""
    if len(points_xy) == 0:
        return []
    pts = np.array(points_xy)
    transformed = np.column_stack([pts[:, 0], pts[:, 1] * Y_WEIGHT])
    db = DBSCAN(eps=eps_x_equivalent, min_samples=1).fit(transformed)
    clusters = []
    for label in set(db.labels_):
        member_idx = np.where(db.labels_ == label)[0]
        clusters.append(pts[member_idx])
    return clusters

def cluster_hull(cluster_pts, buffer_yd=1.5):
    """Real 2-D convex hull for >=3 non-collinear points; a small buffered
    point/line for degenerate 1-2 point clusters (can't form a hull)."""
    if len(cluster_pts) >= 3:
        mp = MultiPoint([tuple(p) for p in cluster_pts])
        hull = mp.convex_hull
        if hull.geom_type == "Polygon":
            return hull
        return hull.buffer(buffer_yd)
    elif len(cluster_pts) == 2:
        return LineString([tuple(cluster_pts[0]), tuple(cluster_pts[1])]).buffer(buffer_yd)
    else:
        return Point(tuple(cluster_pts[0])).buffer(buffer_yd)

def carry_path_crosses_hulls(start, end, opp_points, eps):
    """Full 2-D pipeline for one carry: cluster real opponent positions into lines,
    build each line's convex hull, and count how many hulls the carry's actual
    2-D path (not just its x-range) geometrically intersects. Forward-progress
    only: 'breaking a line' means advancing through it, not grazing a defender's
    zone while moving sideways or backward (matches the cited tactical definition
    and the direction restriction used in the earlier 1-D method)."""
    if len(opp_points) == 0 or end[0] <= start[0]:
        return 0, 0
    clusters = cluster_lines_2d(opp_points, eps)
    path = LineString([tuple(start), tuple(end)])
    crossed = 0
    for c in clusters:
        hull = cluster_hull(c)
        if path.intersects(hull):
            crossed += 1
    return crossed, len(clusters)

def voronoi_control_gained(start, end, teammate_pts, opponent_pts, carrier_start, grid_step=2.0, pad=6.0):
    """Static, position-only Voronoi pitch control (no velocity data available from
    a single freeze frame, so this approximates -- not replicates -- dynamic models
    like Spearman et al.). Grid restricted to a corridor around the carry's own path
    so the metric reflects local space gained, not the whole pitch."""
    lo_x, hi_x = min(start[0], end[0]) - pad, max(start[0], end[0]) + pad
    lo_y, hi_y = min(start[1], end[1]) - pad, max(start[1], end[1]) + pad
    lo_x, hi_x = max(0, lo_x), min(120, hi_x)
    lo_y, hi_y = max(0, lo_y), min(80, hi_y)
    xs = np.arange(lo_x, hi_x, grid_step)
    ys = np.arange(lo_y, hi_y, grid_step)
    if len(xs) == 0 or len(ys) == 0:
        return 0.0
    gx, gy = np.meshgrid(xs, ys)
    grid_pts = np.column_stack([gx.ravel(), gy.ravel()])

    all_before = np.array(teammate_pts + opponent_pts + [carrier_start])
    n_team_before = len(teammate_pts) + 1  # + carrier
    team_before_idx = set(range(n_team_before))

    all_after = np.array(teammate_pts + opponent_pts + [end])  # carrier moved to end location
    team_after_idx = team_before_idx  # same index layout

    tree_before = cKDTree(all_before)
    _, nn_before = tree_before.query(grid_pts)
    control_before = np.mean([1 if i in team_before_idx else 0 for i in nn_before])

    tree_after = cKDTree(all_after)
    _, nn_after = tree_after.query(grid_pts)
    control_after = np.mean([1 if i in team_after_idx else 0 for i in nn_after])

    return float(control_after - control_before)

"""
with open("spatial_lib.py", "w") as f:
    f.write(spatial_lib_source)

import spatial_lib
from spatial_lib import cluster_lines_2d, cluster_hull, carry_path_crosses_hulls, voronoi_control_gained
print("spatial_lib.py written and imported. Default Y_WEIGHT =", spatial_lib.Y_WEIGHT)


## Step 5 — 2-D grid search (`grid_search_2d.py`)

**2-D upgrade validation.** Grid-searches the two spatial parameters (lateral-distance
weight, clustering gap) for the genuine 2-D method, restricted to tactically realistic line
counts (thesis Table 3.5).

Requires `base_carries_s1.pkl` from Step 2. Writes `spatial_grid_search.json`; the winning
combination (`y_weight=0.35, eps=6`) is already saved at `data/spatial_params_final.json`
and is what Step 6 uses.


In [ ]:
# NOTE: this cell expects base_carries_s1.pkl from Step 2 and will error without it.
import pandas as pd, numpy as np, json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import spatial_lib
from spatial_lib import carry_path_crosses_hulls

df = pd.read_pickle("base_carries_s1.pkl")
print(f"Base carries: {len(df)}")

feat_cols = ["carry_distance_m", "carry_speed_ms", "defender_proximity_m", "under_pressure"]
idx = np.arange(len(df))
idx_train, idx_temp = train_test_split(idx, test_size=0.4, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42)

y_weights = [0.15, 0.25, 0.35]
eps_grid = [4, 6, 8, 10, 12]

results = []
X = df[feat_cols].values

for yw in y_weights:
    spatial_lib.Y_WEIGHT = yw
    for eps in eps_grid:
        targets, n_lines_list = [], []
        for _, row in df.iterrows():
            crossed, n_lines = carry_path_crosses_hulls(row["start"], row["end"], row["opp_pts"], eps)
            targets.append(int(crossed >= 1))
            n_lines_list.append(n_lines)
        y = np.array(targets)
        rate = y.mean()
        avg_lines = np.mean(n_lines_list)

        Xtr, ytr = X[idx_train], y[idx_train]
        Xval, yval = X[idx_val], y[idx_val]
        gb = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42)
        gb.fit(Xtr, ytr)
        val_auc = roc_auc_score(yval, gb.predict_proba(Xval)[:, 1])

        results.append({"y_weight": yw, "eps": eps, "avg_n_lines": round(avg_lines,2),
                         "line_break_rate": round(100*rate,2), "val_auc": round(val_auc,4)})
        print(f"y_weight={yw}  eps={eps}  avg_lines={avg_lines:.2f}  rate={100*rate:.2f}%  val_AUC={val_auc:.4f}", flush=True)

best = max(results, key=lambda r: r["val_auc"])
print(f"\nBest combo by validation AUC: {best}")
json.dump({"grid": results, "best": best}, open("spatial_grid_search.json", "w"), indent=2)


## Step 6 — Main pipeline (`parse_final.py`)

**Main pipeline.** Re-parses all real match files using the validated 2-D parameters:
computes progressive-carry flags, 2-D line-breaking, static pitch control, and multi-phase
(possession-chain) downstream xG. This is the single source of truth for all reported
numbers.

Requires `events/`+`threesixty/` (Season 1) and `season2/events/`+`season2/threesixty/`
(Season 2) from Step 1. The output is already provided at `data/carries_final.csv`, so this
step is only needed if you're regenerating from scratch.


In [ ]:
# NOTE: this cell expects both seasons' events/threesixty from Step 1 and will error without them.
import json, glob, math, csv
import spatial_lib
from spatial_lib import carry_path_crosses_hulls, voronoi_control_gained

spatial_lib.Y_WEIGHT = 0.35   # grid-search-validated, forward-progress-only version
EPS_2D = 6                     # grid-search-validated

MID_ATT_POSITIONS = {
    "Left Center Midfield", "Right Center Midfield", "Center Defensive Midfield",
    "Left Defensive Midfield", "Right Defensive Midfield", "Left Midfield", "Right Midfield",
    "Center Attacking Midfield", "Left Attacking Midfield", "Right Attacking Midfield",
    "Left Wing", "Right Wing", "Center Forward", "Left Center Forward", "Right Center Forward",
}
YARDS_TO_M = 0.9144
GOAL = (120.0, 40.0)
FINAL_40_X = 120 * 0.6
QUICK_TRANSITION_SEC = 6.0  # "counter-press window" heuristic for chaining possessions

def dist_yd(a, b):
    return math.hypot(a[0]-b[0], a[1]-b[1])
def dist_to_goal_yd(pt):
    return dist_yd(pt, GOAL)
def ts_to_sec(ts):
    h, m, s = ts.split(":")
    return int(h)*3600 + int(m)*60 + float(s)

def build_possession_chains(events):
    """Group possession segments into chains: consecutive spells by the same team
    stay in one chain even through a brief (<QUICK_TRANSITION_SEC) opponent spell,
    modelling a loose-ball/counter-press regain rather than a genuine reset."""
    segments = []  # (period, possession, team_id, start_idx, end_idx, start_sec, end_sec)
    cur = None
    for e in events:
        key = (e.get("period"), e.get("possession"))
        team_id = e.get("possession_team", {}).get("id")
        if team_id is None:
            continue
        try:
            t = ts_to_sec(e["timestamp"])
        except Exception:
            continue
        if cur is None or (cur["period"], cur["possession"]) != key:
            if cur is not None:
                segments.append(cur)
            cur = {"period": e.get("period"), "possession": e.get("possession"), "team_id": team_id,
                   "start_idx": e["index"], "end_idx": e["index"], "start_sec": t, "end_sec": t}
        else:
            cur["end_idx"] = e["index"]
            cur["end_sec"] = t
    if cur is not None:
        segments.append(cur)

    chain_id_by_possession = {}
    last_team_segment = {}  # team_id -> (chain_id, end_sec) most recent segment for that team
    next_chain_id = 0
    for seg in segments:
        t = seg["team_id"]
        if t in last_team_segment:
            prev_chain, prev_end_sec = last_team_segment[t]
            gap = seg["start_sec"] - prev_end_sec
            if seg["period"] == segments[0]["period"] or True:  # same-period check applied via timestamps resetting; gap covers it
                pass
            if gap <= QUICK_TRANSITION_SEC and gap >= 0:
                chain_id = prev_chain
            else:
                chain_id = next_chain_id; next_chain_id += 1
        else:
            chain_id = next_chain_id; next_chain_id += 1
        chain_id_by_possession[(seg["period"], seg["possession"])] = chain_id
        last_team_segment[t] = (chain_id, seg["end_sec"])
    return chain_id_by_possession

def parse_season(events_dir, threesixty_dir, season_tag):
    rows = []
    for ef in sorted(glob.glob(f"{events_dir}/*.json")):
        match_id = ef.split("/")[-1].replace(".json", "")
        events = json.load(open(ef))
        try:
            frames = json.load(open(f"{threesixty_dir}/{match_id}.json"))
            frame_by_event = {f["event_uuid"]: f for f in frames}
        except FileNotFoundError:
            frame_by_event = {}

        chain_by_poss = build_possession_chains(events)

        # shots grouped by (team_id, chain_id) -> list of (index, xg)
        shots_by_chain = {}
        for e in events:
            if e["type"]["name"] == "Shot" and "shot" in e:
                key_poss = (e.get("period"), e.get("possession"))
                chain_id = chain_by_poss.get(key_poss)
                team_id = e.get("team", {}).get("id")
                shots_by_chain.setdefault((team_id, chain_id), []).append(
                    (e["index"], e["shot"].get("statsbomb_xg", 0.0))
                )
            # also same-possession only (for comparison / backward compatibility)

        shots_by_possession = {}
        for e in events:
            if e["type"]["name"] == "Shot" and "shot" in e:
                key = (e.get("possession"), e.get("team", {}).get("id"))
                shots_by_possession.setdefault(key, []).append((e["index"], e["shot"].get("statsbomb_xg", 0.0)))

        for e in events:
            if e["type"]["name"] != "Carry":
                continue
            pos = e.get("position", {}).get("name")
            if pos not in MID_ATT_POSITIONS:
                continue
            if "location" not in e or "carry" not in e or "end_location" not in e["carry"]:
                continue
            start, end = e["location"], e["carry"]["end_location"]
            duration = e.get("duration")
            if duration is None or duration <= 0:
                continue
            dist_yards = dist_yd(start, end)
            dist_m = dist_yards * YARDS_TO_M
            speed_ms = dist_m / duration
            if dist_yards < 3.0 or duration < 0.4 or speed_ms > 12.0:
                continue

            d_start_goal_m = dist_to_goal_yd(start) * YARDS_TO_M
            d_end_goal_m = dist_to_goal_yd(end) * YARDS_TO_M
            progression_m = d_start_goal_m - d_end_goal_m
            threshold_m = 5.0 if start[0] >= FINAL_40_X else 10.0
            progressive_carry = int(progression_m >= threshold_m)

            frame = frame_by_event.get(e["id"])
            defender_prox_m = None
            lines_broken_2d = None
            n_opponents = None
            pitch_control_gained = None
            if frame and frame.get("freeze_frame"):
                opp_pts = [pl["location"] for pl in frame["freeze_frame"] if not pl.get("teammate", True) and not pl.get("keeper", False)]
                team_pts = [pl["location"] for pl in frame["freeze_frame"] if pl.get("teammate", True) and not pl.get("actor", False)]
                n_opponents = len(opp_pts)
                if opp_pts:
                    defender_prox_m = min(dist_yd(start, o) * YARDS_TO_M for o in opp_pts)
                    crossed, _ = carry_path_crosses_hulls(start, end, opp_pts, EPS_2D)
                    lines_broken_2d = crossed
                    pitch_control_gained = voronoi_control_gained(start, end, team_pts, opp_pts, start)

            # ---- downstream value: single-possession (old) vs multi-phase chain (new) ----
            key_single = (e.get("possession"), e.get("team", {}).get("id"))
            downstream_xg_single = sum(xg for idx, xg in shots_by_possession.get(key_single, []) if idx > e["index"])

            key_poss = (e.get("period"), e.get("possession"))
            chain_id = chain_by_poss.get(key_poss)
            team_id = e.get("team", {}).get("id")
            downstream_xg_chain = sum(xg for idx, xg in shots_by_chain.get((team_id, chain_id), []) if idx > e["index"])

            rows.append({
                "season": season_tag, "match_id": match_id,
                "player": e["player"]["name"], "player_id": e["player"]["id"],
                "team": e["team"]["name"], "position": pos,
                "period": e["period"], "minute": e["minute"],
                "start_x": start[0], "start_y": start[1], "end_x": end[0], "end_y": end[1],
                "carry_distance_m": round(dist_m, 4), "carry_duration_s": round(duration, 4),
                "carry_speed_ms": round(speed_ms, 4),
                "under_pressure": int(bool(e.get("under_pressure", False))),
                "progressive_carry": progressive_carry,
                "defender_proximity_m": round(defender_prox_m, 4) if defender_prox_m is not None else "",
                "n_opponents_in_frame": n_opponents if n_opponents is not None else "",
                "lines_broken_2d": lines_broken_2d if lines_broken_2d is not None else "",
                "line_break_2d": int(lines_broken_2d >= 1) if lines_broken_2d is not None else "",
                "pitch_control_gained": round(pitch_control_gained, 5) if pitch_control_gained is not None else "",
                "downstream_xg_single_poss": round(downstream_xg_single, 4),
                "downstream_xg_chain": round(downstream_xg_chain, 4),
            })
    return rows

# rows = []
# rows += parse_season("events", "threesixty", "LaLiga_2020_21")
# rows += parse_season("season2/events", "season2/threesixty", "Ligue1_2021_22")
# print(f"Total qualifying carries: {len(rows)}")
# with open(os.path.join(DATA_DIR, "carries_final.csv"), "w", newline="") as f:
#     w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
#     w.writeheader()
#     w.writerows(rows)
# print("Saved carries_final.csv")


## Step 7 — Formal statistics (`stats_inference.py`)

**Formal statistics.** OLS (robust SE) for pitch-control-gained, logistic regression for
line-breaking (6-feature baseline), player fixed-effects logistic regression with full
p-values/CIs (both seasons), and bootstrap 95% CIs for classifier AUC (thesis Tables
4.3–4.7).

This step reads `data/carries_final.csv`, which is already provided — **this is the first
step that will actually run end-to-end without any extra downloads.**


In [ ]:
import pandas as pd, numpy as np, json
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

np.random.seed(42)
df = pd.read_csv(os.path.join(DATA_DIR, "carries_final.csv"))
df["line_break_2d"] = pd.to_numeric(df["line_break_2d"], errors="coerce")
df["progressive_carry"] = df["progressive_carry"].astype(int)
df["under_pressure"] = df["under_pressure"].astype(int)
df = df.dropna(subset=["defender_proximity_m", "line_break_2d", "pitch_control_gained"]).copy()
df["line_break_2d"] = df["line_break_2d"].astype(int)

FEAT6 = ["carry_distance_m", "carry_duration_s", "carry_speed_ms", "defender_proximity_m", "under_pressure", "progressive_carry"]

ROSTER1 = ['Lionel Andrés Messi Cuccittini','Frenkie de Jong','Pedro González López','Sergio Busquets i Burgos',
           'Ousmane Dembélé','Antoine Griezmann','Philippe Coutinho Correia','Miralem Pjanić',
           'Vinícius José Paixão de Oliveira Júnior','Toni Kroos','Marcos Llorente Moreno','João Félix Sequeira',
           'Luka Modrić','Iago Aspas Juncal','Karim Benzema']
SHORT1 = {'Lionel Andrés Messi Cuccittini':'Messi','Frenkie de Jong':'de Jong','Pedro González López':'Pedri',
    'Sergio Busquets i Burgos':'Busquets','Ousmane Dembélé':'Dembélé','Antoine Griezmann':'Griezmann',
    'Philippe Coutinho Correia':'Coutinho','Miralem Pjanić':'Pjanić',
    'Vinícius José Paixão de Oliveira Júnior':'Vinícius Jr','Toni Kroos':'Kroos','Marcos Llorente Moreno':'Llorente',
    'João Félix Sequeira':'João Félix','Luka Modrić':'Modrić','Iago Aspas Juncal':'Aspas','Karim Benzema':'Benzema'}
ROSTER2 = ['Lionel Andrés Messi Cuccittini','Kylian Mbappé Lottin','Neymar da Silva Santos Junior',
           'Ángel Fabián Di María Hernández','Marco Verratti','Idrissa Gana Gueye','Danilo Luís Hélio Pereira',
           'Georginio Wijnaldum','Dimitri Payet','Seko Fofana']
SHORT2 = {'Lionel Andrés Messi Cuccittini':'Messi (PSG)','Kylian Mbappé Lottin':'Mbappé','Neymar da Silva Santos Junior':'Neymar',
          'Ángel Fabián Di María Hernández':'Di María','Marco Verratti':'Verratti','Idrissa Gana Gueye':'Gueye',
          'Danilo Luís Hélio Pereira':'Danilo Pereira','Georginio Wijnaldum':'Wijnaldum','Dimitri Payet':'Payet','Seko Fofana':'Fofana'}

def coef_table(fit, names):
    ci = fit.conf_int()
    out = []
    for n in names:
        out.append({
            "term": n, "coef": round(float(fit.params[n]), 4), "se": round(float(fit.bse[n]), 4),
            "z_or_t": round(float(fit.tvalues[n]), 3), "p_value": round(float(fit.pvalues[n]), 4),
            "ci_low": round(float(ci.loc[n, 0]), 4), "ci_high": round(float(ci.loc[n, 1]), 4),
            "significant_95": bool(fit.pvalues[n] < 0.05),
        })
    return out

results = {}

# ===================== 1. OLS: pitch_control_gained ~ 6 features (continuous outcome, robust SE) =====================
for season in df["season"].unique():
    d = df[df["season"] == season]
    X = sm.add_constant(d[FEAT6])
    y = d["pitch_control_gained"]
    fit = sm.OLS(y, X).fit(cov_type="HC3")  # heteroskedasticity-robust SEs
    tab = coef_table(fit, ["const"] + FEAT6)
    results[f"ols_pitch_control_{season}"] = {"r_squared": round(fit.rsquared, 4), "n": int(fit.nobs), "table": tab}
    print(f"\n=== OLS pitch_control_gained ~ 6 features [{season}] (HC3 robust SE) ===")
    print(f"R^2 = {fit.rsquared:.4f}  n = {int(fit.nobs)}")
    for row in tab:
        print(f"  {row['term']:<24s} coef={row['coef']:+.4f}  SE={row['se']:.4f}  p={row['p_value']:.4f}  "
              f"95% CI=[{row['ci_low']:+.4f}, {row['ci_high']:+.4f}]  {'*' if row['significant_95'] else ''}")

# ===================== 2. Logit: line_break_2d ~ 6 features (baseline, both seasons) =====================
for season in df["season"].unique():
    d = df[df["season"] == season]
    X = sm.add_constant(d[FEAT6])
    y = d["line_break_2d"]
    fit = sm.Logit(y, X).fit(disp=0)
    tab = coef_table(fit, ["const"] + FEAT6)
    results[f"logit_baseline_{season}"] = {"pseudo_r2": round(fit.prsquared, 4), "n": int(fit.nobs), "table": tab}
    print(f"\n=== Logit line_break_2d ~ 6 features [{season}] ===")
    print(f"Pseudo R^2 = {fit.prsquared:.4f}  n = {int(fit.nobs)}")
    for row in tab:
        print(f"  {row['term']:<24s} coef={row['coef']:+.4f}  SE={row['se']:.4f}  p={row['p_value']:.4f}  "
              f"95% CI=[{row['ci_low']:+.4f}, {row['ci_high']:+.4f}]  {'*' if row['significant_95'] else ''}")

# ===================== 3. Logit with player fixed effects, formal inference (both seasons) =====================
def fixed_effects_inference(season_tag, roster, short_map, min_n=10):
    d = df[df["season"] == season_tag].copy()
    d["short_name"] = d["player"].map(short_map)
    d = d[d["player"].isin(roster)]
    counts = d["short_name"].value_counts()
    keep = counts[counts >= min_n].index.tolist()
    d = d[d["short_name"].isin(keep)]
    baseline_player = sorted(keep)[0]
    dummies = pd.get_dummies(d["short_name"], prefix="player", drop_first=True).astype(int)
    X = pd.concat([d[FEAT6].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
    X = sm.add_constant(X)
    y = d["line_break_2d"].reset_index(drop=True)
    fit = sm.Logit(y, X).fit(disp=0, maxiter=200)
    player_terms = [c for c in X.columns if c.startswith("player_")]
    tab = coef_table(fit, player_terms)
    for row in tab:
        row["player"] = row["term"].replace("player_", "")
        row["n"] = int(counts[row["player"]])
    print(f"\n=== Logit + player fixed effects [{season_tag}] (baseline: {baseline_player}) ===")
    print(f"Pseudo R^2 = {fit.prsquared:.4f}  n = {int(fit.nobs)}")
    for row in sorted(tab, key=lambda r: -r["coef"]):
        sig = "***" if row["p_value"]<0.01 else ("**" if row["p_value"]<0.05 else ("*" if row["p_value"]<0.10 else ""))
        print(f"  {row['player']:<16s} coef={row['coef']:+.4f}  SE={row['se']:.4f}  p={row['p_value']:.4f}  "
              f"95% CI=[{row['ci_low']:+.4f}, {row['ci_high']:+.4f}]  n={row['n']:<5d} {sig}")
    return {"baseline_player": baseline_player, "pseudo_r2": round(fit.prsquared, 4), "n": int(fit.nobs), "table": tab}

results["fe_season1"] = fixed_effects_inference("LaLiga_2020_21", ROSTER1, SHORT1)
results["fe_season2"] = fixed_effects_inference("Ligue1_2021_22", ROSTER2, SHORT2)

# ===================== 4. Bootstrap CI for GBM AUC (no classical parameters to test) =====================
def bootstrap_auc(d, feat_cols, target, n_boot=300):
    X, y = d[feat_cols].values, d[target].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    gb = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42)
    gb.fit(Xtr, ytr)
    proba = gb.predict_proba(Xte)[:, 1]
    point_auc = roc_auc_score(yte, proba)
    n = len(yte)
    rng = np.random.RandomState(42)
    boot_aucs = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if len(np.unique(yte[idx])) < 2:
            continue
        boot_aucs.append(roc_auc_score(yte[idx], proba[idx]))
    boot_aucs = np.array(boot_aucs)
    return {"point_auc": round(float(point_auc), 4),
            "boot_mean": round(float(boot_aucs.mean()), 4),
            "ci_low": round(float(np.percentile(boot_aucs, 2.5)), 4),
            "ci_high": round(float(np.percentile(boot_aucs, 97.5)), 4)}

for season in df["season"].unique():
    d = df[df["season"] == season]
    r5 = bootstrap_auc(d, FEAT6[:5], "line_break_2d")
    r6 = bootstrap_auc(d, FEAT6, "line_break_2d")
    results[f"gbm_boot_5feat_{season}"] = r5
    results[f"gbm_boot_6feat_{season}"] = r6
    print(f"\n=== GBM bootstrap AUC 95% CI [{season}] ===")
    print(f"  5-feat: AUC={r5['point_auc']}  95% CI=[{r5['ci_low']}, {r5['ci_high']}]")
    print(f"  6-feat: AUC={r6['point_auc']}  95% CI=[{r6['ci_low']}, {r6['ci_high']}]")

json.dump(results, open(os.path.join(DATA_DIR, "inference_results.json"), "w"), indent=2)
print("\nSaved inference_results.json")


## Step 8 — Dashboard v2 (`make_dashboard_v2.py`)

Season 1 descriptive dashboard (speed distribution, line-break rates, feature importance) —
thesis Figure 4.1.

**Heads up:** this script reads `carries_full_v2.csv`, `player_stats_v2.csv`, and
`model_results_v2.json` — files from an earlier draft of the pipeline that are **not**
included in `data/` (the provided outputs only cover the final, corrected pipeline:
`carries_final.csv`, `inference_results.json`, `spatial_params_final.json`,
`gap_grid_search.json`). Regenerating this exact figure would require reconstructing those
intermediate v2 files. The code is kept as-is below for reference; update the paths to
match your own intermediate files if you want to run it.


In [ ]:
import pandas as pd, numpy as np, json
import matplotlib.pyplot as plt

df = pd.read_csv("carries_full_v2.csv")
stats = pd.read_csv("player_stats_v2.csv")
results = json.load(open("model_results_v2.json"))

plt.style.use("dark_background")
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Ball-Carrying Kinematics v2 — Refined Definitions, Real StatsBomb Data", fontsize=14.5, fontweight="bold", color="#ffffff", y=0.995)
fig.text(0.5, 0.965, "Progressive carry (The Football Analyst / StatsBomb-FBref def.) + line-clustering line-break metric", ha="center", va="center", fontsize=10, color="#cbd5e1")
fig.text(0.5, 0.935, "REAL MATCH DATA — StatsBomb Open Data (events + 360 freeze frames)", ha="center", va="center", fontsize=10, fontweight="bold", color="#7ee787",
          bbox=dict(boxstyle="round,pad=0.35", facecolor="#0d2818", edgecolor="#2ea043"))

featured = df[df["is_featured"]]
rest = df[~df["is_featured"]]

# Panel 1: carry speed distribution (unchanged, still real baseline)
ax1 = axes[0, 0]
ax1.hist(rest["carry_speed_ms"], bins=40, alpha=0.5, color="#64748b", label=f"Rest of sample (n={len(rest):,})", density=True)
ax1.hist(featured["carry_speed_ms"], bins=40, alpha=0.65, color="#00ff87", label=f"Featured 15 players (n={len(featured):,})", density=True)
ax1.set_title("1. Average Carry Speed — Real Distribution", fontsize=11, fontweight="bold", color="#e2e8f0", pad=10)
ax1.set_xlabel("Average carry speed, distance/duration (m/s)", fontsize=9, color="#94a3b8")
ax1.set_ylabel("Density", fontsize=9, color="#94a3b8")
ax1.legend(facecolor="#0b132b", edgecolor="#1e293b", labelcolor="white", fontsize=8)
ax1.grid(True, linestyle="--", alpha=0.15, color="#e2e8f0")

# Panel 2: mass-sensitivity check -- KE under position-based vs flat mass model
ax2 = axes[0, 1]
lims = [0, max(df["ke_mid_j"].max(), df["ke_flat_j"].max()) * 1.03]
ax2.plot(lims, lims, color="#f59e0b", linestyle="--", linewidth=1, alpha=0.8, label="y = x (no mass effect)")
ax2.scatter(rest["ke_flat_j"], rest["ke_mid_j"], color="#64748b", alpha=0.12, s=8, edgecolor="none")
ax2.scatter(featured["ke_flat_j"], featured["ke_mid_j"], color="#00ff87", alpha=0.35, s=14, edgecolor="none")
ax2.set_xlim(lims); ax2.set_ylim(lims)
ax2.set_title("2. Mass-Sensitivity Check: KE Proxy", fontsize=11, fontweight="bold", color="#e2e8f0", pad=10)
ax2.set_xlabel("KE using flat 80kg mass (J)", fontsize=9, color="#94a3b8")
ax2.set_ylabel("KE using position-based mass (J)", fontsize=9, color="#94a3b8")
ax2.legend(facecolor="#0b132b", edgecolor="#1e293b", labelcolor="white", fontsize=8, loc="upper left")
ax2.grid(True, linestyle="--", alpha=0.15, color="#e2e8f0")

# Panel 3: refined line-break rate by player
ax3 = axes[1, 0]
plot_stats = stats.sort_values("line_break_rate_v2_pct", ascending=True)
colors = ["#10b981" if pl != "Rest of sample (pooled)" else "#f59e0b" for pl in plot_stats["player"]]
ax3.barh(plot_stats["player"], plot_stats["line_break_rate_v2_pct"], color=colors, edgecolor="#334155", height=0.65)
ax3.axvline(stats.loc[stats["player"]=="Rest of sample (pooled)","line_break_rate_v2_pct"].values[0],
            color="#f59e0b", linestyle="--", linewidth=1, alpha=0.7)
ax3.set_title("3. Refined Line-Break Rate (Defensive-Line Clustering)", fontsize=10.5, fontweight="bold", color="#e2e8f0", pad=10)
ax3.set_xlabel("% of carries crossing \u22651 clustered opponent line", fontsize=8.5, color="#94a3b8")
ax3.tick_params(axis="y", labelsize=8)
ax3.grid(True, linestyle="--", alpha=0.15, color="#e2e8f0", axis="x")

# Panel 4: feature importance, model 2b (with progressive_carry)
ax4 = axes[1, 1]
imp = results["model2b_importances"]
labels_map = {"carry_distance_m":"Carry distance","carry_duration_s":"Carry duration","carry_speed_ms":"Carry speed",
              "defender_proximity_m":"Defender proximity","under_pressure":"Under pressure (flag)","progressive_carry":"Progressive carry (flag)"}
names = [labels_map[k] for k in imp.keys()]
vals = list(imp.values())
order = np.argsort(vals)
names = [names[i] for i in order]; vals = [vals[i] for i in order]
bars = ax4.barh(names, vals, color=["#3b82f6" if n!="Progressive carry (flag)" else "#f59e0b" for n in names], edgecolor="#1d4ed8", height=0.55)
ax4.set_title(f"4. Feature Importance incl. Progressive Carry (AUC={results['model2b_auc']:.3f})", fontsize=10, fontweight="bold", color="#e2e8f0", pad=10)
ax4.set_xlabel("Relative importance (%) — gradient boosting, refined target", fontsize=8.5, color="#94a3b8")
for bar in bars:
    w = bar.get_width()
    ax4.text(w + 1, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va="center", ha="left", color="#ffffff", fontsize=8, fontweight="bold")
ax4.set_xlim(0, max(vals) + 15)
ax4.grid(True, linestyle="--", alpha=0.15, color="#e2e8f0")

for ax in axes.flat:
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#1e293b"); ax.spines["bottom"].set_color("#1e293b")

fig.text(0.5, 0.012,
          "Source: StatsBomb open data, La Liga 2020/21. Progressive carry def.: The Football Analyst / StatsBomb-FBref. "
          "Line-break: opponent-line clustering, refined from a simpler bypass count.",
          ha="center", va="center", fontsize=8, color="#94a3b8", style="italic")

plt.tight_layout(rect=[0, 0.03, 1, 0.90])
plt.savefig("real_kinematic_dashboard_v2.png", dpi=300)
print("saved real_kinematic_dashboard_v2.png")


## Step 9 — Dashboard v3 (`make_dashboard_v3.py`)

Grid-search curve, xG-linkage comparison, Season 2 replication, AUC comparison (addendum
figure).

**Heads up:** like Step 8, this script reads `results_v3.json` and
`player_stats_season2.csv`, which aren't among the provided final outputs (only
`gap_grid_search.json` is available in `data/`, and is used here). Kept as-is for
reference; adjust paths to your own generated files to run it.


In [ ]:
import os
import pandas as pd, numpy as np, json
import matplotlib.pyplot as plt

results = json.load(open("results_v3.json"))
grid = json.load(open(os.path.join(DATA_DIR, "gap_grid_search.json")))
s2_stats = pd.read_csv("player_stats_season2.csv")

plt.style.use("dark_background")
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Ball-Carrying Kinematics v3 — Grid Search, xG Linkage, Season-2 Replication", fontsize=14, fontweight="bold", color="#ffffff", y=0.995)
fig.text(0.5, 0.965, "La Liga 2020/21 + Ligue 1 2021/22 — Real StatsBomb Event & 360° Data", ha="center", va="center", fontsize=10.5, color="#cbd5e1")
fig.text(0.5, 0.935, "REAL MATCH DATA — StatsBomb Open Data (two seasons, two leagues)", ha="center", va="center", fontsize=10, fontweight="bold", color="#7ee787",
          bbox=dict(boxstyle="round,pad=0.35", facecolor="#0d2818", edgecolor="#2ea043"))

# Panel 1: grid search curve
ax1 = axes[0, 0]
g = pd.DataFrame(grid["grid"])
ax1.plot(g["gap_yd"], g["val_auc"], marker="o", color="#00ff87", linewidth=2, markersize=7)
ax1.axvline(grid["chosen_gap_yd"], color="#f59e0b", linestyle="--", linewidth=1.2, label=f"Chosen gap = {grid['chosen_gap_yd']} yd")
ax1.set_title("1. Line-Clustering Gap: Grid Search", fontsize=11, fontweight="bold", color="#e2e8f0", pad=10)
ax1.set_xlabel("Clustering gap (yards)", fontsize=9, color="#94a3b8")
ax1.set_ylabel("Validation AUC (line-break classifier)", fontsize=9, color="#94a3b8")
ax1.legend(facecolor="#0b132b", edgecolor="#1e293b", labelcolor="white", fontsize=8)
ax1.grid(True, linestyle="--", alpha=0.15, color="#e2e8f0")

# Panel 2: xG linkage — progressive vs line-break
ax2 = axes[0, 1]
cats = ["Not\nprogressive", "Progressive", "No line\nbreak", "Line\nbreak"]
vals = [results["xg_combined"]["xg_not_prog"], results["xg_combined"]["xg_prog"],
        results["xg_combined"]["xg_no_break"], results["xg_combined"]["xg_break"]]
colors = ["#64748b", "#f59e0b", "#64748b", "#10b981"]
bars = ax2.bar(cats, vals, color=colors, edgecolor="#334155", width=0.6)
ax2.set_title("2. Downstream Shot Value (xG) — Combined, Real", fontsize=11, fontweight="bold", color="#e2e8f0", pad=10)
ax2.set_ylabel("Avg. xG from a shot later in same possession", fontsize=8.5, color="#94a3b8")
for bar in bars:
    h = bar.get_height()
    ax2.text(bar.get_x()+bar.get_width()/2, h+0.001, f"{h:.3f}", ha="center", color="white", fontsize=9, fontweight="bold")
ax2.grid(True, linestyle="--", alpha=0.15, color="#e2e8f0", axis="y")

# Panel 3: season-2 player line-break / progressive rates
ax3 = axes[1, 0]
plot_stats = s2_stats.sort_values("line_break_pct", ascending=True)
colors3 = ["#10b981" if pl != "Rest of sample" else "#f59e0b" for pl in plot_stats["player"]]
ax3.barh(plot_stats["player"], plot_stats["line_break_pct"], color=colors3, edgecolor="#334155", height=0.65)
ax3.set_title("3. Ligue 1 (PSG) Line-Break Rate — Real, Out-of-Sample", fontsize=10, fontweight="bold", color="#e2e8f0", pad=10)
ax3.set_xlabel("% of carries crossing \u22651 clustered opponent line", fontsize=8.5, color="#94a3b8")
ax3.tick_params(axis="y", labelsize=8)
ax3.grid(True, linestyle="--", alpha=0.15, color="#e2e8f0", axis="x")

# Panel 4: AUC replication across seasons
ax4 = axes[1, 1]
labels4 = ["S1: 5 feat", "S1: +progressive", "S2: 5 feat", "S2: +progressive"]
vals4 = [results["auc_s1_5feat"], results["auc_s1_6feat"], results["auc_s2_5feat"], results["auc_s2_6feat"]]
colors4 = ["#3b82f6", "#f59e0b", "#3b82f6", "#f59e0b"]
bars4 = ax4.bar(labels4, vals4, color=colors4, edgecolor="#1d4ed8", width=0.6)
ax4.set_title("4. Model Replication: La Liga vs Ligue 1", fontsize=11, fontweight="bold", color="#e2e8f0", pad=10)
ax4.set_ylabel("ROC-AUC (line-break classifier)", fontsize=9, color="#94a3b8")
ax4.set_ylim(0.6, 0.85)
for bar in bars4:
    h = bar.get_height()
    ax4.text(bar.get_x()+bar.get_width()/2, h+0.005, f"{h:.3f}", ha="center", color="white", fontsize=9, fontweight="bold")
ax4.grid(True, linestyle="--", alpha=0.15, color="#e2e8f0", axis="y")

for ax in axes.flat:
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#1e293b"); ax.spines["bottom"].set_color("#1e293b")

fig.text(0.5, 0.012, "Source: StatsBomb open data. Season 1 = Barcelona's La Liga 2020/21; Season 2 = PSG's Ligue 1 2021/22 (Messi, Mbapp\u00e9, Neymar).",
          ha="center", va="center", fontsize=8, color="#94a3b8", style="italic")

plt.tight_layout(rect=[0, 0.03, 1, 0.90])
plt.savefig("dashboard_v3.png", dpi=300)
print("saved dashboard_v3.png")


## Step 10 — Spatial example figure (`make_spatial_example.py`)

Renders one real carry's convex hulls and Voronoi pitch-control grid (thesis Figure
3.2/3.3).

Requires `example_carry.pkl` — a single carry row (`{"start": [...], "end": [...],
"opp_pts": [...]}`) pickled from the base-carries table built in Step 2. Not included in
`data/`; if you have `base_carries_s1.pkl` from Step 2, you can build one, e.g.:

```python
import pickle
df1 = pd.read_pickle("base_carries_s1.pkl")
pickle.dump(df1.iloc[0].to_dict(), open("example_carry.pkl", "wb"))
```


In [ ]:
import pickle, numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
import spatial_lib
spatial_lib.Y_WEIGHT = 0.35
from spatial_lib import cluster_lines_2d, cluster_hull, voronoi_control_gained
from scipy.spatial import cKDTree

row = pickle.load(open("example_carry.pkl", "rb"))
start, end = row["start"], row["end"]
opp_pts = row["opp_pts"]

plt.style.use("dark_background")
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# --- Panel 1: pitch, opponents, clusters, hulls, carry path ---
ax = axes[0]
ax.set_facecolor("#0b3d0b")
ax.add_patch(plt.Rectangle((0, 0), 120, 80, fill=False, edgecolor="white", linewidth=1.5))
ax.plot([60, 60], [0, 80], color="white", linewidth=0.8, alpha=0.5)

clusters = cluster_lines_2d(opp_pts, 6)
colors = plt.cm.autumn(np.linspace(0.1, 0.9, len(clusters)))
for c, col in zip(clusters, colors):
    hull = cluster_hull(c)
    if hull.geom_type == "Polygon":
        xs, ys = hull.exterior.xy
        ax.add_patch(MplPolygon(list(zip(xs, ys)), closed=True, facecolor=col, alpha=0.35, edgecolor=col, linewidth=2))
    ax.scatter(c[:, 0], c[:, 1], color=col, s=90, edgecolor="white", zorder=5)

ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="-|>", color="#00ff87", lw=3))
ax.scatter([start[0]], [start[1]], color="#00ff87", s=140, marker="o", edgecolor="white", zorder=6, label="Carry start")
ax.scatter([end[0]], [end[1]], color="#00ff87", s=140, marker="s", edgecolor="white", zorder=6, label="Carry end")
ax.set_xlim(0, 120); ax.set_ylim(0, 80)
ax.set_title(f"2-D Line-Break Detection: Convex Hulls\n({len(clusters)} lines detected, real 360\u00b0 opponent positions)", fontsize=11, fontweight="bold", color="#e2e8f0")
ax.set_xlabel("Attacking direction \u2192 (yards)", fontsize=9, color="#94a3b8")
ax.legend(loc="upper left", facecolor="#0b132b", edgecolor="#1e293b", labelcolor="white", fontsize=8)
ax.set_aspect("equal")

# --- Panel 2: Voronoi pitch control heatmap around the carry ---
ax2 = axes[1]
ax2.set_facecolor("#0b3d0b")
pad = 8
lo_x, hi_x = max(0, min(start[0], end[0]) - pad), min(120, max(start[0], end[0]) + pad)
lo_y, hi_y = max(0, min(start[1], end[1]) - pad), min(80, max(start[1], end[1]) + pad)
xs = np.arange(lo_x, hi_x, 1.0); ys = np.arange(lo_y, hi_y, 1.0)
gx, gy = np.meshgrid(xs, ys)
grid_pts = np.column_stack([gx.ravel(), gy.ravel()])

team_pts = [[start[0]-8, start[1]+6], [start[0]-4, start[1]-10], [start[0]+2, start[1]+14]]  # illustrative teammates
all_after = np.array(team_pts + opp_pts + [end])
team_idx = set(range(len(team_pts) + 1))
tree = cKDTree(all_after)
_, nn = tree.query(grid_pts)
control = np.array([1 if i in team_idx else 0 for i in nn]).reshape(gx.shape)

ax2.contourf(gx, gy, control, levels=[-0.5, 0.5, 1.5], colors=["#7f1d1d", "#14532d"], alpha=0.55)
ax2.scatter([p[0] for p in opp_pts], [p[1] for p in opp_pts], color="#ef4444", s=70, edgecolor="white", zorder=5, label="Opponents")
ax2.scatter([p[0] for p in team_pts], [p[1] for p in team_pts], color="#22c55e", s=70, edgecolor="white", zorder=5, label="Teammates (illustrative)")
ax2.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="-|>", color="#00ff87", lw=3))
ax2.scatter([start[0]], [start[1]], color="#facc15", s=140, marker="o", edgecolor="white", zorder=6, label="Carrier start\u2192end")
ax2.scatter([end[0]], [end[1]], color="#facc15", s=140, marker="s", edgecolor="white", zorder=6)
ax2.set_xlim(lo_x, hi_x); ax2.set_ylim(lo_y, hi_y)
ax2.set_title("Static Voronoi Pitch Control (after carry)\nGreen = attacking team's nearest-player control", fontsize=11, fontweight="bold", color="#e2e8f0")
ax2.set_xlabel("Attacking direction \u2192 (yards)", fontsize=9, color="#94a3b8")
ax2.legend(loc="upper left", facecolor="#0b132b", edgecolor="#1e293b", labelcolor="white", fontsize=7.5)
ax2.set_aspect("equal")

plt.tight_layout()
plt.savefig("spatial_example.png", dpi=200)
print("saved spatial_example.png")


## Step 11 — Forest plot (`make_forest_plot.py`)

Player fixed-effects coefficients with 95% confidence intervals, both seasons (thesis
Figure 4.2).

Reads `data/inference_results.json` — either the provided file, or the one you generated
in Step 7. **This step also runs end-to-end out of the box.**


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

res = json.load(open(os.path.join(DATA_DIR, "inference_results.json")))

plt.style.use("dark_background")
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
fig.patch.set_facecolor("#0b132b")

for ax, key, title, baseline in [
    (axes[0], "fe_season1", "La Liga 2020/21\n(baseline: Aspas)", "Aspas"),
    (axes[1], "fe_season2", "Ligue 1 2021/22\n(baseline: Danilo Pereira)", "Danilo Pereira"),
]:
    ax.set_facecolor("#111827")
    tab = sorted(res[key]["table"], key=lambda r: r["coef"])
    names = [r["player"] for r in tab]
    coefs = [r["coef"] for r in tab]
    los = [r["coef"] - r["ci_low"] for r in tab]
    his = [r["ci_high"] - r["coef"] for r in tab]
    sig = [r["p_value"] < 0.05 for r in tab]
    colors = ["#10b981" if s else "#64748b" for s in sig]
    y_pos = np.arange(len(names))
    for i in range(len(names)):
        ax.errorbar([coefs[i]], [y_pos[i]], xerr=[[los[i]], [his[i]]], fmt="o", color="white",
                    ecolor=colors[i], elinewidth=2.5, capsize=4, markersize=7,
                    markerfacecolor="white", zorder=3)
    ax.axvline(0, color="#f59e0b", linestyle="--", linewidth=1.2, alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names, fontsize=10)
    ax.set_xlabel("Log-odds coefficient vs. baseline (95% CI)", fontsize=9, color="#94a3b8")
    ax.set_title(title, fontsize=12, fontweight="bold", color="#e2e8f0")
    ax.grid(True, linestyle="--", alpha=0.15, color="#e2e8f0", axis="x")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("Player Fixed Effects with 95% Confidence Intervals (Formal Hypothesis Testing)",
             fontsize=14, fontweight="bold", color="white", y=1.04)
fig.text(0.5, -0.03,
          "Green = statistically significant at p<0.05 (crosses zero = not distinguishable from baseline player). "
          "Season 1: no player reaches significance. Season 2: Messi (PSG) is the one significant positive effect.",
          ha="center", fontsize=9.5, color="#cbd5e1", style="italic")

plt.tight_layout()
plt.savefig("forest_plot_fixed_effects.png", dpi=200, bbox_inches="tight")
print("saved forest_plot_fixed_effects.png")
